# BARAM 2026 — EDA 체크리스트

팀 첫 미팅 전/후에 **아래 체크 항목**을 순서대로 실행하세요.

## 체크리스트
- [ ] 1. 라벨 분포·결측·계절성 (`train_labels`)
- [ ] 2. 설비용량 대비 이용률 (10% 평가 마스크 확인)
- [ ] 3. SCADA 파워커브 (학습·이해용, test 입력 금지)
- [ ] 4. LDAPS/GFS 격자 구조·풍속 상관
- [ ] 5. `data_available_kst_dtm` 규칙 확인
- [ ] 6. 격자→KPX 그룹 집계 방식 비교 (`nearest` vs `mean`)
- [ ] 7. 로컬 Score (1-NMAE + FICR) sanity check

> 평가 공식 최종 검증: [데이콘 평가 산식 코드](https://dacon.io/competitions/official/236727/overview/evaluation)

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import GROUP_CAPACITY_KWH, GROUP_COLUMNS, PATHS
from src.data_loader import load_group_centroids, load_labels, load_turbine_info, load_weather
from src.features import aggregate_weather_to_groups, build_group_frame
from src.metrics import evaluate_submission

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
pd.options.display.max_columns = 30

## 1. 라벨 EDA (`train_labels.csv`)

In [ ]:
labels = load_labels()
print('rows:', len(labels))
print('period:', labels['kst_dtm'].min(), '~', labels['kst_dtm'].max())
labels.describe()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
for ax, col in zip(axes, GROUP_COLUMNS):
    labels.plot(x='kst_dtm', y=col, ax=ax, legend=False)
    ax.set_title(col)
plt.tight_layout()

In [ ]:
# 결측·0 발전 비율
summary = []
for col in GROUP_COLUMNS:
    s = labels[col]
    cap = GROUP_CAPACITY_KWH[col]
    summary.append({
        'group': col,
        'missing_ratio': s.isna().mean(),
        'zero_ratio': (s.fillna(0) == 0).mean(),
        'active_ratio_ge_10pct': (s >= cap * 0.10).mean(),
        'utilization_median': (s / cap).median(),
    })
pd.DataFrame(summary)

## 2. 터빈 메타 (`info.xlsx`) + 격자 centroid

In [ ]:
turbines = load_turbine_info()
centroids = load_group_centroids()
display(turbines)
display(centroids)

## 3. SCADA 파워커브 (학습 기간만)

In [ ]:
vestas = pd.read_csv(PATHS['scada_vestas'], encoding='utf-8-sig', nrows=50000)
ws_cols = [c for c in vestas.columns if c.endswith('_ws')]
pw_cols = [c for c in vestas.columns if 'power' in c]

sample = vestas.melt(id_vars=['kst_dtm'], value_vars=ws_cols, var_name='turbine', value_name='ws')
sample_pw = vestas.melt(id_vars=['kst_dtm'], value_vars=pw_cols, var_name='turbine', value_name='power')
sample['turbine_key'] = sample['turbine'].str.replace('_ws', '')
sample_pw['turbine_key'] = sample_pw['turbine'].str.replace('_power_kw10m', '')
curve = sample.merge(sample_pw, on=['kst_dtm', 'turbine_key']).dropna()

plt.figure(figsize=(8, 5))
plt.hexbin(curve['ws'], curve['power'], gridsize=40, cmap='viridis', mincnt=1)
plt.xlabel('wind speed (SCADA)')
plt.ylabel('power (kW10m)')
plt.title('VESTAS power curve (sample)')
plt.colorbar(label='count')

## 4. 기상 데이터 구조 (LDAPS / GFS)

In [ ]:
ldaps = load_weather('ldaps', 'train')
gfs = load_weather('gfs', 'train')

print('LDAPS rows:', len(ldaps), '| grids per timestamp:', ldaps.groupby('forecast_kst_dtm').size().median())
print('GFS rows:', len(gfs), '| grids per timestamp:', gfs.groupby('forecast_kst_dtm').size().median())

ldaps.head(3)

In [ ]:
# data_available 규칙: 같은 날 01~24시는 동일한 available 시각인지
check = ldaps.copy()
check['forecast_date'] = check['forecast_kst_dtm'].dt.date
n_unique = check.groupby('forecast_date')['data_available_kst_dtm'].nunique()
print('dates with !=1 unique available time:', (n_unique != 1).sum())
check.groupby('forecast_date')['data_available_kst_dtm'].first().head()

## 5. 격자 집계 + 라벨 상관

In [ ]:
ldaps_grp = aggregate_weather_to_groups(ldaps, source='ldaps', method='nearest')
frame = build_group_frame(ldaps_grp, labels=labels)

corr_cols = [c for c in frame.columns if c.startswith('ldaps_ws') or c == 'power_kwh']
corr = frame[corr_cols].corr(numeric_only=True)['power_kwh'].sort_values(ascending=False)
corr.head(10)

## 6. 로컬 Score sanity check (2024 hold-out)

In [ ]:
valid_start = pd.Timestamp('2024-01-01 01:00:00')
train_part = frame[frame['forecast_kst_dtm'] < valid_start].dropna(subset=['power_kwh'])
valid_part = frame[frame['forecast_kst_dtm'] >= valid_start]

# 아주 단순 베이스라인: 풍속-제곱 비례
coef = {}
for gid in [1, 2, 3]:
    p = train_part[train_part['group_id'] == gid]
    x = p['ldaps_ws10'].fillna(0).to_numpy() ** 2
    y = p['power_kwh'].to_numpy()
    denom = np.dot(x, x) + 1e-6
    coef[gid] = float(np.dot(x, y) / denom)

pred_rows = []
for gid in [1, 2, 3]:
    p = valid_part[valid_part['group_id'] == gid].copy()
    p['pred'] = coef[gid] * (p['ldaps_ws10'].fillna(0) ** 2)
    pred_rows.append(p)
pred_long = pd.concat(pred_rows)
pred_wide = pred_long.pivot(index='forecast_kst_dtm', columns='group_id', values='pred')
pred_wide = pred_wide.rename(columns={1: 'kpx_group_1', 2: 'kpx_group_2', 3: 'kpx_group_3'}).reset_index()

true_wide = labels[labels['kst_dtm'] >= valid_start]
scores = evaluate_submission(true_wide, pred_wide, time_col='kst_dtm')
pd.Series(scores)

## 7. 미팅 메모 (팀이 직접 작성)

- 격자 집계 최종안:
- LDAPS / GFS / 둘 다:
- 로컬 검증 구간:
- 다음 실험: